# FRED Chroma Query

In [4]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
from collections import Counter
from datetime import datetime, timezone


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from apps.agentic.core.constants import FRED_COLLECTION_NAME, FRED_DB_NAME
from apps.agentic.core.document_loaders.fred_document_loader import FREDChromaDocumentLoader
from apps.agentic.core.utils import (set_chatgpt_env, set_langsmith_env, set_tavily_env, 
                                     set_github_env)

DB_PATH = str(Path("../../.db").resolve())

set_chatgpt_env(filedir="../../.keys")
set_langsmith_env(filedir="../../.keys")

## Verify Document Counts

In [5]:
FRED_DB_NAME, FRED_COLLECTION_NAME, DB_PATH, os.listdir(DB_PATH)

('fred',
 'fred',
 '/Users/troy/Develop/gly.fish/yada/.db',
 ['.DS_Store', 'research_library', 'fred', 'pdf_document_library', 'github'])

In [6]:
vs = FREDChromaDocumentLoader(DB_PATH).vectorstore
coll = vs._collection

In [7]:
client = vs._client
print([c.name for c in client.list_collections()])
print("Opened:", coll.name)
print("total docs:", coll.count())

['fred']
Opened: fred
total docs: 13028


## Verify Document Metadata

In [19]:
probe = coll.get(limit=15000)
metadata_list = probe.get("metadatas") if probe else []
metas = [m for m in metadata_list or [] if m]
len(metas)

13028

In [20]:
keys = set().union(*(m.keys() for m in metas)) if metas else set()
for key in sorted(keys):
    print(key)

category_id
category_name
category_path
filename
frequency
last_updated
leaf_name
observation_end
observation_start
popularity
seasonal_adjustment
series_id
series_title
units


## Search Examples 

In [22]:
vs = FREDChromaDocumentLoader(DB_PATH).vectorstore
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
    },
)

hits = retriever.invoke("Which Price series track Farm commodities?")
[print(h) for h in hits]

page_content='# Producer Price Index by Commodity: Farm Products

## Series Information
- **Series ID:** WPU01
- **Frequency:** Monthly
- **Units:** Index 1982=100
- **Seasonal Adjustment:** Not Seasonally Adjusted
- **Observation Range:** 1913-01-01 – 2025-09-01
- **Last Updated:** 2025-11-25T09:47:19-06:00
- **Popularity:** 42

## Category Information
- **Category ID:** 33528
- **Category Name:** Farm Products
- **Leaf Name:** Farm Products
- **Category Path:** Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products' metadata={'category_name': 'Farm Products', 'popularity': 42, 'leaf_name': 'Farm Products', 'category_path': 'Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products', 'series_title': 'Producer Price Index by Commodity: Farm Products', 'observation_end': '2025-09-01', 'category_id': 33528, 'seasonal_adjustment': 'Not Seasonally Adjusted', 'filename': 'fred_prices_32455.yaml', 'last_updated': '2025-11-25T09:47:19-06:00', 'series_id': 'WPU01

[None, None, None, None, None, None, None, None]

## Metadata Queries

In [23]:
category_counts = Counter(m.get("category_name") for m in metas if "category_name" in m and m.get("category_name"))
for category, count in category_counts.most_common(50):
    print(f"{count:5d}  {category}")

 1000  Commodities
 1000  Manufacturing
  768  Machinery and Equipment
  726  House Price Indexes
  516  Metals and Metal Products
  484  Processed Foods and Feeds
  417  Employment Cost Index
  327  Retail Trade
  306  Intermediate Demand By Production Flow
  305  Chemicals and Allied Products
  297  By Harmonized System
  291  End Use Classification System
  288  By NAICS
  255  Farm Products
  252  Textile Products and Apparel
  245  Transportation and Warehousing
  242  Standard International Trade Classification
  241  Pulp, Paper, and Allied Products
  228  Miscellaneous Products
  227  Information
  225  Inputs to Industries
  224  Nonmetallic Mineral Products
  220  Final Demand
  211  Special Indexes
  211  Health Care and Social Assistance
  207  Fuels and Related Products and Power
  202  Furniture and Household Durables
  201  Wholesale Trade
  194  Transportation Equipment
  191  Mining, Quarrying, and Oil and Gas Extraction
  139  Lumber and Wood Products
  127  Finance a

### Series

In [15]:
probe = coll.get(
    where={"category_name": "Farm Products"},
    limit=200
)
metadata_list = probe.get("metadatas") if probe else []
names = [m["series_title"] for m in metadata_list or [] if m]

for name in names:
    print(name)

Producer Price Index by Commodity: Farm Products (DISCONTINUED)
Producer Price Index by Commodity: Farm Products: Fruits and Melons, Fresh/Dry Vegetables and Nuts (DISCONTINUED)
Producer Price Index by Commodity: Farm Products: Fresh Fruits and Melons
Producer Price Index by Commodity: Farm Products: Citrus Fruits
Producer Price Index by Commodity: Farm Products: Other Fruits and Berries
Producer Price Index by Commodity: Farm Products: Fresh and Dry Vegetables (DISCONTINUED)
Producer Price Index by Commodity: Farm Products: Dry Vegetables
Producer Price Index by Commodity: Farm Products: Fresh Vegetables, Except Potatoes
Producer Price Index by Commodity: Farm Products: Sweet Potatoes
Producer Price Index by Commodity: Farm Products: Potatoes
Producer Price Index by Commodity: Farm Products: Grains
Producer Price Index by Commodity: Farm Products: Wheat (DISCONTINUED)
Producer Price Index by Commodity: Farm Products: Wheat (DISCONTINUED)
Producer Price Index by Commodity: Farm Product

In [16]:
paths = [m["category_path"] for m in metadata_list or [] if m]
for path in paths:
    print(path)

Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm P